# Space & Time

In the first module, we studied numerical integration methods for the solution of ordinary differential equations (ODEs), using the phugoid model of glider flight as a motivation. In this module, we will study the numerical solution of *partial differential equations (PDEs)*, where the unknown is a multi-variate function. The problem could depend on time, $t$, and one spatial dimension $x$ (or more), which means we need to build a discretization grid with each independent variable.

We will start our discussion of numerical PDEs with 1D linear and non-linear convection equations, the 1D diffusion equation, and 1D Burgers' equation.

## 1D linear convection

The *one-dimensional linear convection equation* is the simplest, most basic model that can be used to learn something about numerical solution of PDEs. It's surprising that this little equation can teach us so much! Here it is:

$$
\label{eq-linear-convection-pde}
\frac{\partial u}{\partial t} + c \frac{\partial u}{\partial x} = 0
$$

The equation represents a *wave* propagating with speed $c$ in the $x$ direction, without change of shape. For that reason, it's sometimes called the *one-way wave equation* (sometimes also the *advection equation*).

With an initial condition $u(x,0)=u_0(x)$, the equation has an exact solution given by:

$$
\label{eq-linear-convection-exact-solution}
u(x,t)=u_0(x-ct)
$$

Go on: check it. Take the time and space derivative and stick them into the equation to see that it holds.

Look at the exact solution for a moment ... we know two things about it: 

1. its shape does not change, being always the same as the initial wave, $u_0$, only shifted in the $x$-direction; and 
2. it's constant along so-called **characteristic curves**, $x-ct=$constant. This means that for any point in space and time, you can move back along the characteristic curve to $t=0$ to know the value of the solution.

```{figure} ./figures/characteristics.png
:label: fig-linear-convection-characteristics
:alt: Parallel characteristic curves rising through a space-time diagram for a positive convection speed
:width: 400px
:align: center

Characteristic curves $x-ct=\text{constant}$ for a positive convection speed $c$.
```

Why do we call the equations *linear*? PDEs can be either linear or non-linear. In a linear equation, the unknown function $u$ and its derivatives appear only in linear terms, in other words, there are no products, powers, or transcendental functions applied on them. 

The most important feature of linear equations is: solutions can be _superposed_ to generate new solutions that still satisfy the original equation. This is super useful!

## Finite-differences

In the previous lessons, we discretized time derivatives; now we have derivatives in both space *and* time, so we need to discretize with respect to *both* these variables. 

Imagine a *space-time* plot, where the coordinates in the vertical direction represent advancing in time—for example, from $t^n$ to $t^{n+1}$—and the coordinates in the horizontal direction move in space: consecutive points are $x_{i-1}$, $x_i$, and $x_{i+1}$.  This creates a grid where a point has both a temporal and spatial index. Here is a graphical representation of the space-time grid:

$$
\label{eq-space-time-grid}
\begin{matrix}
t^{n+1} & \rightarrow & \bullet  && \bullet  && \bullet  \\
t^n & \rightarrow & \bullet  && \bullet  && \bullet  \\
& &  x_{i-1} && x_i && x_{i+1}
\end{matrix}
$$

For the numerical solution of $u(x,t)$, we'll use subscripts to denote the spatial position, like $u_i$, and superscripts to denote the temporal instant, like $u^n$.  We would then label the solution at the top-middle point in the grid above as follows:
$u^{n+1}_{i}$.

Each grid point below has an subscript index, corresponding to the spatial position and increasing to the right, and an superscript index, corresponding to the time instant and increasing upwards.  A small grid segment would have the following values of the numerical solution at each point:

$$
\label{eq-space-time-indexed-values}
\begin{matrix}
& &\bullet & & \bullet & &  \bullet \\
& &u^{n+1}_{i-1} & & u^{n+1}_i & & u^{n+1}_{i+1} \\
& &\bullet & & \bullet & &  \bullet \\
& &u^n_{i-1} & & u^n_i & & u^n_{i+1} \\
& &\bullet & & \bullet & &  \bullet \\
& &u^{n-1}_{i-1} & & u^{n-1}_i & & u^{n-1}_{i+1} \\
\end{matrix}
$$

Another way to explain our discretization grid is to say that it is built with constant steps in time and space, $\Delta t$ and $\Delta x$, as follows:

$$
\label{eq-uniform-space-time-grid}
\begin{aligned}
x_i &= i\, \Delta x \quad \text{and} \quad t^n= n\, \Delta t, \\
u_i^n &= u(i\, \Delta x, n\, \Delta t).
\end{aligned}
$$

### Discretizing our model equation

Let's see how to discretize the 1D linear convection equation in both space and time.  By definition, the partial derivative with respect to time differentiates only with time and not with space; its discretized form changes only the $n$ indices.  Similarly, the partial derivative with respect to $x$ differentiates with space not time, and only the $i$ indices are affected.  

We'll discretize the spatial coordinate $x$ into points indexed from $i=0$ to $N$, and then step in discrete time intervals of size $\Delta t$.

From the definition of a derivative (and simply removing the limit), we know that for $\Delta x$ sufficiently small:

$$
\label{eq-forward-space-difference}
\frac{\partial u}{\partial x}\approx \frac{u(x+\Delta x)-u(x)}{\Delta x}
$$

This formula could be applied at any point $x_i$. But note that it's not the only way that we can estimate the derivative. The geometrical interpretation of the first derivative $\partial u/ \partial x$ at any point is that it represents the slope of the tangent to the curve $u(x)$. In the sketch below, we show a slope line at $x_i$ and mark it as "exact." If the formula written above is applied at $x_i$, it approximates the derivative using the next spatial grid point: it is then called a _forward difference_ formula. 

But as shown in the sketch below, we could also estimate the spatial derivative using the point behind $x_i$, in which case it is called a _backward difference_. We could even use the two points on each side of $x_i$, and obtain what's called a _central difference_ (but in that case the denominator would be $2\Delta x$).

```{figure} ./figures/FDapproxiamtions.png
:label: fig-finite-difference-approximations
:alt: Forward, backward, and central secant slopes compared with the exact tangent slope at x sub i
:width: 300px
:align: center

Forward, backward, and central finite-difference approximations to the slope at $x_i$.
```

We have three possible ways to represent a discrete form of $\partial u/ \partial x$:

* Forward difference: uses $x_i$ and $x_i + \Delta x$,
* Backward difference: uses $x_i$ and $x_i- \Delta x$,
* Central difference: uses two points on either side of $x_i$.

The sketch above also suggests that some finite-difference formulas might be better than others: it looks like the *central difference* approximation is closer to the slope of the "exact" derivative. We'll see later how to make this observation rigorous.

The three formulas are:

$$
\label{eq-first-derivative-difference-formulas}
\begin{aligned}
\frac{\partial u}{\partial x} &\approx \frac{u(x_{i+1})-u(x_i)}{\Delta x} &&\text{Forward},\\
\frac{\partial u}{\partial x} &\approx \frac{u(x_i)-u(x_{i-1})}{\Delta x} &&\text{Backward},\\
\frac{\partial u}{\partial x} &\approx \frac{u(x_{i+1})-u(x_{i-1})}{2\Delta x} &&\text{Central}.
\end{aligned}
$$

Euler's method is equivalent to using a forward-difference scheme for the time derivative. Let's stick with that, and choose the backward-difference scheme for the space derivative.  Our discrete equation is then:

$$
\label{eq-linear-convection-ftbs}
\frac{u_i^{n+1}-u_i^n}{\Delta t} + c \frac{u_i^n - u_{i-1}^n}{\Delta x} = 0
$$

where $n$ and $n+1$ are two consecutive steps in time, while $i-1$ and $i$ are two neighboring points of the discretized $x$ coordinate. With given initial conditions, the only unknown in this discretization is $u_i^{n+1}$.  We solve for this unknown to get an equation that lets us step in time, as follows:

$$
\label{eq-linear-convection-ftbs-update}
u_i^{n+1} = u_i^n - c \frac{\Delta t}{\Delta x}(u_i^n-u_{i-1}^n)
$$

It is useful to sketch a grid segment, showing the grid points that influence our numerical solution. This is called a **stencil**. Below is the stencil for solving our model equation with the finite-difference formula we wrote above.

```{figure} ./figures/FTBS_stencil.png
:label: fig-linear-convection-ftbs-stencil
:alt: Space-time stencil connecting values at i minus 1 and i at time n to the value at i at time n plus 1
:width: 300px
:align: center

Stencil for the forward-time, backward-space discretization of linear convection.
```

### And compute!

Alright. Let's get a little Python on the road. First: we need to load our array and plotting libraries, as usual.

:::{warning .simple .dropdown icon=false open=false} In your notebook

Create a new, clean notebook for your work rather than executing the code cells in this lesson from top to bottom. Keep the lesson open as a reference and reconstruct the calculation there.

Type the grid definition, square-wave initial condition, FTBS update, and time loop yourself so that you can trace the numerical update back to [Equation %s](#eq-linear-convection-ftbs-update). Before running each experiment, record what you expect the wave to do. Reproduce the fixed-`dt` grid experiment and the controlled refinement study later in the lesson; compare the controlled study with the order you derive on paper. In the nonlinear section, implement and check both updates yourself before delegating the comparison to an agent. You may copy mechanical details such as imports and plot formatting when transcription would add no understanding.

Consult [Reconstruct a lesson](../../appendices/notebook-workflow.md#notebook-reconstruct) for the general workflow.
:::

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

We also set notebook-wide plotting parameters for the font family and the font size by modifying entries of the `rcParams` dictionary.

In [ ]:
# Set the font family and size to use for Matplotlib figures.
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

As a first exercise, we'll solve the 1D linear convection equation with a *square wave* initial condition, defined as follows:

$$
\label{eq-square-wave-initial-condition}
u(x,0)=\begin{cases}2 & \text{where } 0.5\leq x \leq 1,\\
1 & \text{everywhere else in } (0, 2)
\end{cases}
$$

We also need a boundary condition on $x$: let $u=1$ at $x=0$. Our spatial domain for the numerical solution will only cover the range $x\in (0, 2)$.

```{figure} ./figures/squarewave.png
:label: fig-square-wave-initial-condition
:alt: Square pulse with value two between x equals one half and one on a background value of one
:width: 340px
:align: center

Square-wave initial condition with a unit background and a pulse of height $2$.
```

Now let's define a few variables; we want to make an evenly spaced grid of points within our spatial domain. In the code below, we define a variable called `nx` that will be the number of spatial grid points, and a variable `dx` that will be the distance between any pair of adjacent grid points. We also can define a step in time, `dt`, a number of steps, `nt`, and a value for the wave speed: we like to keep things simple and make $c=1$.  

In [ ]:
# Set parameters.
nx = 41    # number of spatial discrete points
L = 2.0    # length of the 1D domain
dx = L / (nx - 1)  # spatial grid size
nt = 25    # number of time steps
dt = 0.02  # time-step size
c = 1.0    # convection speed

# Define the grid point coordinates.
x = np.linspace(0.0, L, num=nx)

We also need to set up our initial conditions. Here, we use the NumPy function `np.ones()` defining an array which is `nx`-element long with every value equal to $1$. How useful! We then *change a slice* of that array to the value $u=2$, to get the square wave, and we print out the initial array just to admire it. But which values should we change?  The problem states that we need to change the indices of `u` such that the square wave begins at $x = 0.5$ and ends at $x = 1$.

We can use the [`np.where()`](https://numpy.org/doc/stable/reference/generated/numpy.where.html) function to return a list of indices where the vector $x$ meets some conditions.
The function [`np.logical_and()`](https://numpy.org/doc/stable/reference/generated/numpy.logical_and.html) computes the truth value of `x >= 0.5` **and** `x <= 1.0`, element-wise.

In [ ]:
# Set initial conditions with 1.0 everywhere (for now).
u0 = np.ones(nx)
# Get a list of indices where 0.5 <= x <= 1.0.
mask = np.where(np.logical_and(x >= 0.5, x <= 1.0))
print(mask)

With the list of indices, we can now update our initial conditions to get a square-wave shape.

In [ ]:
# Set initial condition u = 2.0 where 0.5 <= x <= 1.0.
u0[mask] = 2.0
print(u0)

Now let's take a look at those initial conditions we've built with a handy plot.

In [ ]:
# Plot the initial conditions.
plt.figure(figsize=(3.0, 3.0))
plt.title('Initial conditions')
plt.xlabel('x')
plt.ylabel('u')
plt.grid()
plt.plot(x, u0, color='tab:blue', linestyle='--', linewidth=2)
plt.xlim(0.0, L)
plt.ylim(0.0, 2.5);

It does look pretty close to what we expected. But it looks like the sides of the square wave are not perfectly vertical. Is that right? Think for a bit.

Now it's time to write some code for the discrete form of the convection equation using our chosen finite-difference scheme. 

For every element of our array `u`, we need to perform the operation: 

$$
\label{eq-linear-convection-code-update}
u_i^{n+1} = u_i^n - c \frac{\Delta t}{\Delta x}(u_i^n-u_{i-1}^n)
$$

We'll store the result in a new (temporary) array `un`, which will be the solution $u$ for the next time-step.  We will repeat this operation for as many time-steps as we specify and then we can see how far the wave has traveled.  

We first initialize the placeholder array `u` to hold the values we calculate for the $n+1$ time step, beginning with a copy of the initial condition.

Then, we may think we have two iterative operations: one in space and one in time (we'll learn differently later), so we may start by nesting a spatial loop inside the time loop, as shown below. You see that the code for the finite-difference scheme is a direct expression of the discrete equation.

Here, `nt` counts updates rather than stored time levels. Because `range(nt)` starts at zero and performs exactly `nt` passes, the final physical time in this example is `nt * dt`.

In [ ]:
u = u0.copy()
for n in range(nt):
    un = u.copy()
    for i in range(1, nx):
        u[i] = un[i] - c * dt / dx * (un[i] - un[i - 1])

**Note 1**—We stressed above that our physical problem needs a boundary condition at $x=0$. Here we do not need to impose it at every iteration because our discretization does not change the value of u[0]: it remains equal to one and our boundary condition is therefore satisfied during the whole computation!

**Note 2**—We will learn later that the code as written above is quite inefficient, and there are better ways to write this, Python-style. But let's carry on.

Now compare the computed profile with the exact square wave translated to the same physical time.

In [ ]:
# Evaluate the exact translated square wave at the final time.
t_final = nt * dt
u_exact = np.ones_like(x)
exact_mask = np.where(
    np.logical_and(x >= 0.5 + c * t_final,
                   x <= 1.0 + c * t_final)
)
u_exact[exact_mask] = 2.0

# Plot the numerical and exact solutions with the initial condition.
plt.figure(figsize=(3.0, 3.0))
plt.xlabel('x')
plt.ylabel('u')
plt.grid()
plt.plot(x, u0, label='Initial',
            color='tab:blue', linestyle='--', linewidth=1)
plt.plot(x, u_exact, label=f'Exact, t = {t_final:.2f}',
            color='tab:gray', linestyle='--', linewidth=2)
plt.plot(x, u, label=f'FTBS, t = {t_final:.2f}',
            color='tab:red', linestyle='-', linewidth=2)
plt.legend()
plt.xlim(0.0, L)
plt.ylim(0.0, 2.5);

That's funny. Our square wave has definitely moved to the right, but it's no longer in the shape of a top-hat. **What's going on?**

### Does a finer grid always help?

The rounded profile suggests that a finer spatial grid might improve the solution. Test that idea while keeping `dt`, `nt`, and $c$ fixed: only the number of spatial points changes below. Before executing the cell, predict what each refinement will do. Then compare the computed profiles with the exact translated square wave and record whether the numerical values remain between $1$ and $2$.

Treat this as an observation rather than a convergence study, and do not change `dt` to repair any surprising result. We will return to what happens—and why—in the next lesson.

In [ ]:
# Change only the number of spatial points.
nx_trials = [41, 81, 101, 121]
fig, axes = plt.subplots(2, 2, figsize=(7.0, 7.0), sharex=True)

for ax, nx_trial in zip(axes.flat, nx_trials):
    dx_trial = L / (nx_trial - 1)
    x_trial = np.linspace(0.0, L, num=nx_trial)
    u_trial = np.ones(nx_trial)
    pulse = np.where(
        np.logical_and(x_trial >= 0.5, x_trial <= 1.0)
    )
    u_trial[pulse] = 2.0

    for n in range(nt):
        un_trial = u_trial.copy()
        for i in range(1, nx_trial):
            u_trial[i] = (
                un_trial[i]
                - c * dt / dx_trial
                * (un_trial[i] - un_trial[i - 1])
            )

    exact_trial = np.ones(nx_trial)
    exact_pulse = np.where(
        np.logical_and(
            x_trial >= 0.5 + c * t_final,
            x_trial <= 1.0 + c * t_final,
        )
    )
    exact_trial[exact_pulse] = 2.0

    ax.plot(x_trial, exact_trial, color='black',
            linestyle=':', linewidth=2, label='Exact')
    ax.plot(x_trial, u_trial, color='tab:red',
            linewidth=2, label='FTBS')
    ax.set_title(f'nx = {nx_trial}')
    ax.set_xlabel('x')
    ax.set_ylabel('u')
    ax.set_xlim(0.0, L)
    ax.grid()
    print(f'nx = {nx_trial:3d}: '          f'min(u) = {u_trial.min():8.3f}, '          f'max(u) = {u_trial.max():8.3f}')

axes.flat[0].legend()
fig.tight_layout();

## Spatial truncation error

Recall the backward-difference approximation we are using for the spatial derivative:

$$
\label{eq-backward-space-difference-revisited}
\frac{\partial u}{\partial x}\approx \frac{u(x)-u(x-\Delta x)}{\Delta x}
$$

We obtain it by using the definition of the derivative at a point, and simply removing the limit, in the assumption that $\Delta x$ is very small. But we already learned with Euler's method that this introduces an error, called the *truncation error*.

We can determine the spatial order of this error by expanding $u(x_i-\Delta x)$ in a Taylor series about $x_i$. Complete that step on paper before reading the result.

:::{warning .simple .dropdown icon=false open=false} On paper — derive the spatial truncation error

Derive the accuracy of the backward-difference formula independently of the computation:

1. Write the Taylor expansion of $u(x_i-\Delta x)$ about $x_i$, retaining terms through the third spatial derivative.
2. Rearrange the expansion to solve for $\partial u/\partial x$ at $x_i$.
3. Identify the leading term omitted by the backward-difference approximation and state its order in $\Delta x$.
4. Predict the factor by which the spatial error should decrease when $\Delta x$ is halved for a smooth solution.

Keep the derivation beside you when you reach the controlled refinement study. It provides an expected slope that is independent of the numerical experiment.
:::

$$
\label{eq-backward-space-difference-expansion}
\frac{\partial u}{\partial x}(x_i) = \frac{u(x_i)-u(x_{i-1})}{\Delta x} + \frac{\Delta x}{2} \frac{\partial^2 u}{\partial x^2}(x_i) - \frac{\Delta x^2}{6} \frac{\partial^3 u}{\partial x^3}(x_i)+ \cdots
$$

The dominant term that is neglected in the finite-difference approximation is of $\mathcal{O}(\Delta x)$. We also see that the approximation *converges* to the exact derivative as $\Delta x \rightarrow 0$. That's good news!

In summary, the chosen "forward-time/backward space" difference scheme is first-order in both space and time: the truncation errors are $\mathcal{O}(\Delta t, \Delta x)$. We'll come back to this!

### A controlled spatial refinement study

The square wave makes numerical diffusion easy to see, but its jumps do not satisfy the smoothness assumed in the Taylor expansion above. To measure the formal spatial behavior of the scheme, use a smooth Gaussian pulse on the same unit background. Its exact solution is the same profile translated by $c t$.

For each grid, compare the numerical and exact solutions at the same final time with the discrete $L_1$ error in [Equation %s](#eq-linear-convection-l1-error):

$$
\label{eq-linear-convection-l1-error}
E = \Delta x \sum_i \left|u_i-u_{\text{exact}}(x_i)\right|.
$$

When the grid spacing is reduced, estimate the observed order from two consecutive errors with [Equation %s](#eq-linear-convection-observed-order):

$$
\label{eq-linear-convection-observed-order}
p = \frac{\log(E_{\text{coarse}}/E_{\text{fine}})}{\log(\Delta x_{\text{coarse}}/\Delta x_{\text{fine}})}.
$$

Use a very small time step for every spatial grid so that time-discretization error is much smaller than the spatial error over this range. Repeating the study with half that time step provides a sensitivity check: the spatial conclusions should barely change.

In [ ]:
def gaussian_profile(x, center=0.7, width=0.2):
    '''Return a smooth pulse on a unit background.'''
    return 1.0 + np.exp(-((x - center) / width)**2)


def advance_linear_convection(u0, c, dx, dt, num_steps):
    '''Advance linear convection with the FTBS scheme.'''
    u = u0.copy()
    for n in range(num_steps):
        un = u.copy()
        for i in range(1, u.size):
            u[i] = un[i] - c * dt / dx * (un[i] - un[i - 1])
    return u

In [ ]:
# Compare several spatial grids at one physical time.
nx_values = np.array([41, 81, 161, 321])
dt_values = [1.0e-4, 5.0e-5]
t_study = 0.1
center = 0.7
width = 0.2
dx_values = L / (nx_values - 1)
error_results = []

for dt_trial in dt_values:
    num_steps = int(round(t_study / dt_trial))
    errors = []
    for nx_trial, dx_trial in zip(nx_values, dx_values):
        x_trial = np.linspace(0.0, L, num=nx_trial)
        u0_trial = gaussian_profile(x_trial, center, width)
        u_trial = advance_linear_convection(
            u0_trial, c, dx_trial, dt_trial, num_steps
        )
        u_exact_trial = gaussian_profile(
            x_trial, center + c * t_study, width
        )
        error = dx_trial * np.sum(np.abs(u_trial - u_exact_trial))
        errors.append(error)
    error_results.append(errors)

error_results = np.array(error_results)
orders = np.log(
    error_results[0, :-1] / error_results[0, 1:]
) / np.log(dx_values[:-1] / dx_values[1:])

print(' nx       dx      E(dt)     E(dt/2)   order')
for j, (nx_trial, dx_trial) in enumerate(zip(nx_values, dx_values)):
    order_text = '  ---' if j == 0 else f'{orders[j - 1]:5.2f}'
    print(f'{nx_trial:3d}  {dx_trial:8.5f}  '          f'{error_results[0, j]:9.3e}  '          f'{error_results[1, j]:9.3e}  {order_text}')

# Compare the errors with a first-order reference slope.
first_order = (
    error_results[0, -1] * dx_values / dx_values[-1]
)
plt.figure(figsize=(4.0, 3.0))
plt.loglog(dx_values, error_results[0], 'o-', label='dt')
plt.loglog(dx_values, error_results[1], 's--', label='dt / 2')
plt.loglog(dx_values, first_order, ':', color='black',
           label='First-order slope')
plt.xlabel(r'$\Delta x$')
plt.ylabel(r'$L_1$ error')
plt.grid(which='both')
plt.xlim([2e-3, 1e-1])
plt.legend();

The error decreases by nearly a factor of two whenever $\Delta x$ is halved, and the observed order approaches $1$. Halving the already small time step barely changes either the errors or the orders. Together, those observations support the first-order spatial behavior predicted by the truncation-error analysis over this tested grid range.

This controlled result does not explain the failed refinement in the earlier square-wave experiment. Keep that discrepancy in mind for the next lesson.

## Non-linear convection

Let's move on to the non-linear convection equation, using the same methods as before. The 1D convection equation is:

$$
\label{eq-nonlinear-convection-pde}
\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = 0
$$

The only difference with the linear case is that we've replaced the constant wave speed $c$ by the variable speed $u$. The equation is non-linear because now we have a product of the solution and one of its derivatives: the product $u\,\partial u/\partial x$. This changes everything!

We're going to use the same discretization as for linear convection: forward difference in time and backward difference in space. Here is the discretized equation:

$$
\label{eq-nonlinear-convection-ftbs}
\frac{u_i^{n+1}-u_i^n}{\Delta t} + u_i^n \frac{u_i^n-u_{i-1}^n}{\Delta x} = 0
$$

Solving for the only unknown term, $u_i^{n+1}$, gives an equation that can be used to advance in time:

$$
\label{eq-nonlinear-convection-ftbs-update}
u_i^{n+1} = u_i^n - u_i^n \frac{\Delta t}{\Delta x} (u_i^n - u_{i-1}^n)
$$

There is very little that needs to change from the code written so far. In fact, we'll even use the same square-wave initial condition. But let's re-initialize the variable `u` with the initial values, and re-enter the numerical parameters here, for convenience (we no longer need $c$, though).

In [ ]:
# Set parameters.
nx = 41  # number of spatial discrete points
L = 2.0  # length of the 1D domain
dx = L / (nx - 1)  # spatial grid size
nt = 10  # number of time steps
dt = 0.02  # time-step size

x = np.linspace(0.0, L, num=nx)
u0 = np.ones(nx)
mask = np.where(np.logical_and(x >= 0.5, x <= 1.0))
u0[mask] = 2.0

 How does it look?

In [ ]:
# Plot the initial conditions.
plt.figure(figsize=(3.0, 3.0))
plt.title('Initial conditions')
plt.xlabel('x')
plt.ylabel('u')
plt.grid()
plt.plot(x, u0, color='C0', linestyle='--', linewidth=2)
plt.xlim(0.0, L)
plt.ylim(0.0, 2.5);

Changing just one line of code in the solution of linear convection, we are able to now get the non-linear solution: the line that corresponds to the discrete equation now has `un[i]` in the place where before we just had `c`. So you could write something like:

```python
for n in range(nt):  
  un = u.copy() 
  for i in range(1, nx): 
    u[i] = un[i] - un[i]*dt/dx*(un[i]-un[i-1]) 
```

We're going to be more clever than that and use NumPy to update _all_ values of the spatial grid in one fell swoop. We don't really need to write a line of code that gets executed *for each* value of $u$ on the spatial grid. Python can update them all at once! Study the code below, and compare it with the one above. Here is a helpful sketch, to illustrate the array operation—also called a "vectorized" operation—for $u_i-u_{i-1}$.

```{figure} ./figures/vectorizedstencil.png
:label: fig-vectorized-backward-difference
:alt: Offset NumPy slices showing how corresponding entries form backward differences across the interior grid points
:width: 400px
:align: center

Array slices used to evaluate $u_i-u_{i-1}$ across the interior grid points. Adapted from @elhage2015.
```

In [ ]:
# Compute the solution using Euler's method and array slicing.
u = u0.copy()
for n in range(nt):
    u[1:] = u[1:] - dt / dx * u[1:] * (u[1:] - u[:-1])

:::{note} Python refresher — why this in-place slice update is safe
:icon: false
The same array `u` appears on both sides of the assignment, but Python evaluates the complete right-hand side before assigning anything to `u[1:]`. The slices `u[1:]` and `u[:-1]` are views of `u`; the arithmetic performed with those views creates a temporary result array containing every new value. Only then does NumPy write that result into the slice on the left. Each update therefore uses values from the old time level.

An element-by-element loop using `u` directly on both sides would not be equivalent:

```python
for i in range(1, nx):
    u[i] = u[i] - dt / dx * u[i] * (u[i] - u[i - 1])
```

After the first iteration, `u[i - 1]` would already belong to the new time level. The earlier loop avoids this mixing by reading from the copy `un`; the vectorized expression avoids it by completing the right-hand-side arithmetic before assignment.
:::

In [ ]:
# Plot the solution after nt time steps with the initial condition.
t_final = nt * dt
plt.figure(figsize=(3.0, 3.0))
plt.xlabel('x')
plt.ylabel('u')
plt.grid()
plt.plot(x, u0, label='Initial',
            color='tab:blue', linestyle='--', linewidth=2)
plt.plot(x, u, label=f'FTBS, t = {t_final:.2f}',
            color='tab:red', linestyle='-', linewidth=2)
plt.legend()
plt.xlim(0.0, L)
plt.ylim(0.0, 2.5);

Hmm. That's quite interesting: like in the linear case, we see that we have lost the sharp sides of our initial square wave, but there's more. Now, the wave has also lost symmetry! It seems to be lagging on the rear side, while the front of the wave is steepening. Is this another form of numerical error, do you ask? No! It's physics!

(same-equations-different-algorithms)=
## Same Equations, Different Algorithms

The front of the wave steepens because larger values of $u$ move faster than smaller values. As that steepening produces a sharp jump, a choice that looked like ordinary algebra begins to matter numerically.

For a differentiable function, the chain rule gives

$$
\label{eq-nonlinear-convection-chain-rule}
\frac{\partial}{\partial x}\left(\frac{u^2}{2}\right)
= u\frac{\partial u}{\partial x}.
$$

We can therefore write the nonlinear convection equation in **conservative form**:

$$
\label{eq-nonlinear-convection-conservative-form}
\frac{\partial u}{\partial t}
+ \frac{\partial f(u)}{\partial x}=0,
\qquad f(u)=\frac{u^2}{2}.
$$

As long as $u$ is smooth, [Equation %s](#eq-nonlinear-convection-pde) and [Equation %s](#eq-nonlinear-convection-conservative-form) describe the same continuous equation. They do not, however, lead to the same discrete algorithm.

### Discretize the two forms

The update used in the preceding computation applies a backward difference directly to $u\,\partial u/\partial x$; we will call it the **pointwise update**. It is [Equation %s](#eq-nonlinear-convection-ftbs-update). Applying the same forward-time, backward-space pattern to the flux derivative instead gives the **conservative update**:

$$
\label{eq-nonlinear-convection-conservative-update}
u_i^{n+1}=u_i^n
-\frac{\Delta t}{\Delta x}
\left[f(u_i^n)-f(u_{i-1}^n)\right]
=u_i^n-\frac{\Delta t}{2\Delta x}
\left[(u_i^n)^2-(u_{i-1}^n)^2\right].
$$

The difference becomes visible by factoring the flux difference:

$$
\label{eq-quadratic-flux-difference-factorization}
\frac{f(u_i)-f(u_{i-1})}{\Delta x}
=\frac{u_i+u_{i-1}}{2}
\frac{u_i-u_{i-1}}{\Delta x}.
$$

The pointwise update multiplies the backward difference by $u_i$, while the conservative update effectively uses the average $(u_i+u_{i-1})/2$. When neighboring values are close, these multipliers become close under refinement. Across a jump, their difference is not small.

### A balance built into the algorithm

The word *conservative* describes an algebraic property of the discrete update. Sum [Equation %s](#eq-nonlinear-convection-conservative-update) over the updated nodes $i=1,\ldots,N$:

$$
\label{eq-nonlinear-convection-discrete-flux-balance}
\begin{aligned}
\Delta x\sum_{i=1}^{N}
\left(u_i^{n+1}-u_i^n\right)
&=-\Delta t\sum_{i=1}^{N}
\left[f(u_i^n)-f(u_{i-1}^n)\right]\\
&=-\Delta t\left[f(u_N^n)-f(u_0^n)\right].
\end{aligned}
$$

Every interior flux appears once with a plus sign and once with a minus sign, so the sum *telescopes*: only the two boundary fluxes remain. If the endpoint fluxes are equal, the discrete total $\Delta x\sum_i u_i$ is unchanged apart from round-off. The pointwise update does not have this flux-difference structure and does not satisfy the same identity. This cancellation principle is central to conservative methods for nonlinear hyperbolic problems [@leveque2002].

When the solution remains differentiable, the chain rule supports the equivalence of the two continuous forms, and consistent discretizations should approach the same smooth solution. At a jump, the ordinary derivative used in the pointwise argument is no longer defined there, so that chain-rule argument is not enough to determine how the jump propagates. The conservative form retains a meaningful balance across the jump.

This short bridge does not yet develop the integral conservation law, weak solutions, shock speeds, or general numerical fluxes. Those ideas belong in Module 3. For now, the important warning is that algebraically equivalent smooth equations can produce algorithms with materially different behavior once discontinuities enter the computation.

:::{warning .simple .dropdown icon=false open=false} On paper — compare one update

Before writing the two functions below, establish independent expectations:

1. Reproduce the chain-rule step leading to [Equation %s](#eq-nonlinear-convection-conservative-form), then derive [Equation %s](#eq-nonlinear-convection-conservative-update).
2. Let $u^n=[1,2,1]$ and $\Delta t/\Delta x=0.1$, with the left value held fixed. Calculate one complete update using the pointwise formula and one using the conservative formula.
3. Expand the sum in [Equation %s](#eq-nonlinear-convection-discrete-flux-balance) for three updated nodes and cross out the canceling interior fluxes.
4. Predict what both updates should do to a constant array, and which update should satisfy the flux-balance identity to round-off.

Keep the two hand-calculated arrays nearby. They test the formulas and time levels directly, independently of the code.
:::

:::{warning .simple .dropdown icon=false open=false} In your notebook — implement both updates

Preserve your earlier direct computation. Then reconstruct the three small functions below in your notebook. Each step function must use only its arguments, leave its input array unchanged, hold the left boundary value fixed, and return a new array. Keep the names and argument order shown here because the agent-written diagnostic code will call this interface.
:::

In [ ]:
def nonlinear_flux(u):
    '''Return the quadratic flux for nonlinear convection.'''
    return 0.5 * u**2


def pointwise_step(u, dt, dx):
    '''Advance one step using the pointwise form.'''
    u_next = u.copy()
    u_next[1:] = (
        u[1:]
        - dt / dx * u[1:] * (u[1:] - u[:-1])
    )
    return u_next


def conservative_step(u, dt, dx):
    '''Advance one step using differences of the quadratic flux.'''
    u_next = u.copy()
    u_next[1:] = (
        u[1:]
        - dt / dx
        * (nonlinear_flux(u[1:]) - nonlinear_flux(u[:-1]))
    )
    return u_next

Check the functions against your hand calculation before asking an agent to build on them. The constant-state checks exercise a different property, so keep both kinds of evidence.

In [ ]:
u_test = np.array([1.0, 2.0, 1.0])
pointwise_expected = np.array([1.0, 1.8, 1.1])
conservative_expected = np.array([1.0, 1.85, 1.15])

np.testing.assert_allclose(
    pointwise_step(u_test, dt=0.1, dx=1.0),
    pointwise_expected,
)
np.testing.assert_allclose(
    conservative_step(u_test, dt=0.1, dx=1.0),
    conservative_expected,
)
np.testing.assert_allclose(u_test, [1.0, 2.0, 1.0])

constant_test = np.full(5, 1.5)
np.testing.assert_allclose(
    pointwise_step(constant_test, dt=0.1, dx=1.0),
    constant_test,
)
np.testing.assert_allclose(
    conservative_step(constant_test, dt=0.1, dx=1.0),
    constant_test,
)

(nonlinear-algorithm-agent-activity)=
## With an agent: compare the algorithms

The two update functions are short enough to derive, implement, and check yourself. Comparing them over several grids and tracking a flux balance is repetitive work that can usefully be delegated. The agent will draft that experimental driver code, but it will not choose the equations, change the algorithms, execute the code, or decide what the evidence means.

:::{warning .simple .dropdown icon=false open=false} With an agent

Use an agent to add an **unexecuted** comparison driver to your own notebook. Allow it to inspect the notebook and add only the cells specified below. Do not allow changes to `nonlinear_flux()`, `pointwise_step()`, `conservative_step()`, earlier cells, or other files. Do not grant package-installation or network access.

You will inspect every added line before running one new cell at a time. The agent may organize the repetitive experiment; you remain responsible for the numerical specification, execution, audit, and conclusion.
:::

### Record expectations before delegation

In your notebook, record your answers to these questions before attaching it to an agent:

1. What maximum change do you expect after either function advances a constant state by one step?
2. What magnitude do you expect for the conservative scheme's residual in [Equation %s](#eq-nonlinear-convection-discrete-flux-balance)?
3. Does the pointwise update have an algebraic reason to produce the same residual?
4. As the grid is refined, should the two algorithms approach one another more clearly for the smooth hump or for the discontinuous square pulse? Why?

These are hypotheses to test, not outputs that the agent should be instructed to manufacture.

### Use this bounded task brief

Add the following brief as a Markdown cell in your notebook. Read every requirement and resolve any mismatch with your own function names before delegating.

#### Agent algorithm-comparison task brief

- **Goal and scope:** Add a comparison driver that compares my existing `pointwise_step()` and `conservative_step()` functions. Do not implement, rewrite, or repair either method. The comparison concerns discrete behavior only; deriving shock speeds, introducing another scheme, and making the final numerical judgment are out of scope.
- **Problem data:** Use $x\in[0,2]$, `nx_values = [81, 161, 321, 641, 1281]`, and `t_compare = 0.15`. For each grid, start with a target `dt = 0.2 * dx`, round the step count to reach the common final time, and then reset `dt = t_compare / num_steps` so every run ends at exactly the same time. Compare (a) the smooth hump $1+\exp[-((x-0.7)/0.2)^2]$ and (b) the square pulse with background $1$ and value $2$ on $0.5\le x\le1$. Each run must begin from a fresh copy of the same initial array.
- **Required diagnostics:** First report the maximum one-step change produced by each method from a constant array with value $1.5$. Write a reusable runner that accepts a step function and the existing `nonlinear_flux()` separately. For every run, compute the initial and final discrete totals $M=\Delta x\sum_i u_i$ and the cumulative balance residual $R=M_{\text{final}}-M_{\text{initial}}+\sum_n\Delta t[f(u_N^n)-f(u_0^n)]$, using the boundary values from the state *before* each update. Also compute the inter-algorithm difference $D=\Delta x\sum_i|u_i^{\text{pointwise}}-u_i^{\text{conservative}}|$.
- **Required output:** Produce one readable table for each profile with `nx`, `dx`, adjusted `dt`, step count, $D$, both balance residuals, and both initial-to-final total changes. Produce a figure showing both final solutions on the finest grid for each profile and a log-log plot of $D$ against $\Delta x$. Label methods, profiles, axes, and final time. Expose numerical values; do not replace them with pass/fail messages or a conclusion. Use only the NumPy and Matplotlib imports already present.
- **Permissions and stopping rule:** Inspect this notebook and add exactly three Python cells followed by one Markdown cell that records the added functions, experiment, plots, and any assumptions. Do not execute any cell, alter existing cells, write another file, install anything, or use the network. Stop after inserting those four cells and report that the comparison driver is ready for human audit.
- **Done when:** The unexecuted cells contain the constant-state comparison, reusable balance runner, two-profile grid study, required tables and plots, plus the change record. An expected trend is not a completion condition; preserve and expose surprising results.

The brief separates the model and evidence you fixed from the code organization the agent may choose. Compare it with the [minimum sufficient specification](../../appendices/agent-use.md#agent-specification-proportionality) before granting access.

### Invoke the agent

Save your notebook so the attachment contains your checked step functions, independent expectations, and task brief. Attach that notebook to your agent interface and send this short request:

:::{card} Prompt
Read the section titled “Agent algorithm-comparison task brief” in the attached notebook. Add the requested comparison driver and change record exactly as specified. Do not execute any cell or change existing work. Stop when the additions are ready for my audit.
:::

If you do not have access to an agent that can edit a notebook, ask it to return the same four proposed cells in chat, add them yourself only after inspection, and follow the audit below.

### Audit before execution

Inspect the notebook diff or the four inserted cells before running them. Confirm that the code:

- calls your existing step functions without redefining or modifying them;
- starts every method and grid from an independent copy of identical initial data;
- uses the requested grids, profiles, and common physical time, and reports the adjusted `dt` and step count;
- accumulates the boundary-flux term from the old state before each call to the step function;
- uses the plus sign in the residual from [Equation %s](#eq-nonlinear-convection-discrete-flux-balance), with right-boundary flux minus left-boundary flux;
- computes $D$ with the two solutions on the same grid and includes the factor $\Delta x$;
- exposes totals and residuals for both methods rather than checking conservation only for the method named `conservative_step`; and
- changes no earlier cell or out-of-scope artifact.

If the code passes inspection, execute one inserted cell at a time. Recalculate one row's $D$ and residual independently from the stored arrays. A clean plot or an agent statement that the method conserves is not a substitute for that arithmetic.

### Test the diagnostics with known defects

After saving the accepted baseline results, add the two deliberately defective functions below under new names. Do not overwrite the correct functions. The first omits the factor $1/2$ in the intended flux; the second reads a newly updated left neighbor while marching across the array.

```python
def conservative_step_missing_half(u, dt, dx):
    '''Deliberately use the wrong quadratic flux.'''
    u_next = u.copy()
    u_next[1:] = (
        u[1:] - dt / dx * (u[1:]**2 - u[:-1]**2)
    )
    return u_next


def pointwise_step_mixed_levels(u, dt, dx):
    '''Deliberately mix old and new time levels.'''
    u_next = u.copy()
    for i in range(1, u.size):
        u_next[i] = (
            u_next[i]
            - dt / dx * u_next[i]
            * (u_next[i] - u_next[i - 1])
        )
    return u_next
```

Apply the corresponding hand-calculated one-step check and the constant-state check to each defective function. Next, use the balance runner for one step from `u_balance_test = np.array([1.0, 2.0, 1.5])`, with `dt = 0.1` and `dx = 1.0`. Compare the correct conservative step with `conservative_step_missing_half`, passing the intended `nonlinear_flux` to the runner in both cases. The unequal endpoint fluxes make this a discriminating balance test.

Finally, repeat that comparison with the square pulse. Explain why its equal endpoint values can let the balance residual pass even with the incorrectly scaled flux. Record which check catches each defect and which checks still pass. If no check fails, inspect the harness before trusting any baseline conclusion. See [Test the checks](../../appendices/verification-patterns.md#verification-defect-injection) for why this adversarial step matters.

:::{warning .simple .dropdown icon=false open=false} Your verdict

Write a short debrief that answers:

1. Did you accept, revise, or reject the agent's harness? Name any change you made before execution.
2. Did both methods preserve a constant state? Report the measured maximum changes.
3. For each profile on the finest grid, report both balance residuals and explain what [Equation %s](#eq-nonlinear-convection-discrete-flux-balance) allows you to conclude.
4. Describe how $D$ changed under refinement for the smooth hump and square pulse. Does the evidence support treating the algorithms as interchangeable in both cases?
5. Which diagnostic caught each injected defect, and why was no single check sufficient?
6. State what remains unresolved until the later conservation-law lesson; do not claim that this experiment derives a shock speed or proves general correctness.
7. Leave a lightweight [agent record](../../appendices/agent-use.md#agent-record) naming the agent or persona, attached notebook, request, permissions, cells accepted or revised, and verification you performed.
:::

### Completion rubric

This is a **Complete/Revise** checkpoint. The activity is complete only when the specification, code audit, executed evidence, defect challenge, and learner judgment are all visible in your notebook.

| Criterion | Complete | Revise |
| --- | --- | --- |
| Independent work | Both step functions pass hand-calculated and constant-state checks completed before delegation. | The agent writes or repairs a numerical method, or expected values are accepted without independent calculation. |
| Scope and access | The agent adds only the four unexecuted cells permitted by the brief. | Existing work changes, code is executed before audit, or access extends beyond the notebook task. |
| Code audit | Checks common data, fresh state, time matching, flux timing and sign, totals, residuals, and $D$. | Relies on names, plots, or the agent's assurance instead of inspecting the calculation. |
| Evidence | Records the requested tables, plots, independent row calculation, and results from both profiles. | Reports only a visual impression or selected values that hide the comparison. |
| Adversarial check | Runs both injected defects and explains which diagnostics detect or miss each one. | Assumes passing baseline checks are effective without testing them against known faults. |
| Verdict and provenance | Makes a bounded claim, names unresolved theory, and records the agent interaction and human revisions. | Declares a generally correct method from this experiment or omits the agent record. |

## What's next?

The next lesson explains the stability restriction behind the failed fixed-`dt` refinement experiment. Module 3 will return to the conservative form from a control-volume balance, define how conservation laws represent discontinuous solutions, determine shock propagation, and apply those ideas to traffic flow.